# LMP Example 7 - Two-Bus, Piecewise Linear Cost Curves
by Xingpeng Li

Converted to Python/Pyomo by Haoxiang Wan

**Lab Website:** [rpglab.github.io](https://rpglab.github.io)

> **Note (e7):** Each generator has a **piecewise linear (PWL) cost curve** with 3 segments.
> The PWL is modelled by introducing segment variables `Pg_seg[g, s]` (bounded by the segment width)
> whose sum equals total dispatch `Pg[g]`. Because segments are filled cheapest-first by the LP,
> no SOS or integer variables are needed. The LMP at each bus is the dual of the nodal balance constraint.
> Compare with e2 (linear cost) to see how PWL pricing changes nodal prices.

In [ ]:
from pyomo.environ import *
import time

# ─────────────────────────────────────────────────────────────────
# System Data
# ─────────────────────────────────────────────────────────────────
BaseMW    = 100
refBus    = 2
total_load = 100  # MW at bus 2

# Branch data
branch_data = {1: {'from': 1, 'to': 2, 'x': 0.1, 'rate': 50}}

# Generator piecewise cost data
# seg_MW: cumulative breakpoints; seg_cost: marginal cost ($/MWh) per segment
gen_data = {
    1: {'bus': 1, 'Pgmin': 30, 'Pgmax': 80,
        'seg_MW':   [30, 60, 80],           # breakpoints (cumulative)
        'seg_cost': [ 5, 10, 15]},           # marginal cost per segment
    2: {'bus': 2, 'Pgmin': 20, 'Pgmax': 90,
        'seg_MW':   [30, 60, 90],
        'seg_cost': [10, 20, 50]},
}


In [ ]:
# ─────────────────────────────────────────────────────────────────
# Build Pyomo Model  (piecewise-linear cost via segment variables)
# ─────────────────────────────────────────────────────────────────
model = ConcreteModel()

# Sets
model.BUS    = Set(initialize=[1, 2])
model.GEN    = Set(initialize=gen_data.keys())
model.BRANCH = Set(initialize=branch_data.keys())

# Compute segment widths and build segment index set
seg_widths = {}   # {(g, s): MW width of segment s}
seg_costs  = {}   # {(g, s): $/MWh}
for g, d in gen_data.items():
    bpts = d['seg_MW']
    for s, (bp, c) in enumerate(zip(bpts, d['seg_cost']), start=1):
        seg_widths[(g, s)] = bp - (bpts[s-2] if s > 1 else 0)
        seg_costs[(g, s)]  = c

n_seg = max(s for (g, s) in seg_widths)
model.SEG = Set(initialize=range(1, n_seg + 1))

# Parameters
model.gen_bus    = Param(model.GEN, initialize={g: gen_data[g]['bus']   for g in gen_data}, within=model.BUS)
model.gen_min    = Param(model.GEN, initialize={g: gen_data[g]['Pgmin'] for g in gen_data})
model.gen_max    = Param(model.GEN, initialize={g: gen_data[g]['Pgmax'] for g in gen_data})
model.seg_width  = Param(model.GEN, model.SEG, initialize=seg_widths)
model.seg_cost   = Param(model.GEN, model.SEG, initialize=seg_costs)
model.branch_x   = Param(model.BRANCH, initialize={k: branch_data[k]['x']    for k in branch_data})
model.branch_rate = Param(model.BRANCH, initialize={k: branch_data[k]['rate'] for k in branch_data})

# Decision Variables
model.Pg     = Var(model.GEN,                  doc='Total dispatch (MW)')
model.Pg_seg = Var(model.GEN, model.SEG,        within=NonNegativeReals, doc='Dispatch in segment s (MW)')
model.pk     = Var(model.BRANCH,               doc='Branch flow (MW)')
model.theta  = Var(model.BUS,                  doc='Voltage angle (rad)')
model.dual   = Suffix(direction=Suffix.IMPORT)

# Objective: minimise piecewise cost
def obj_rule(m):
    return sum(m.seg_cost[g, s] * m.Pg_seg[g, s]
               for g in m.GEN for s in m.SEG)
model.obj = Objective(rule=obj_rule, sense=minimize)

# Total dispatch = sum of segments
def genSegSum_rule(m, g):
    return m.Pg[g] == sum(m.Pg_seg[g, s] for s in m.SEG)
model.genSegSum = Constraint(model.GEN, rule=genSegSum_rule)

# Segment upper bound: Pg_seg[g,s] <= width of segment s
def segLimit_rule(m, g, s):
    return m.Pg_seg[g, s] <= m.seg_width[g, s]
model.segLimit = Constraint(model.GEN, model.SEG, rule=segLimit_rule)

# Generator total limits
def genLimits_rule(m, g):
    return inequality(m.gen_min[g], m.Pg[g], m.gen_max[g])
model.genLimits = Constraint(model.GEN, rule=genLimits_rule)

# Branch thermal limits
def branchLimits_rule(m, k):
    return inequality(-m.branch_rate[k], m.pk[k], m.branch_rate[k])
model.branchLimits = Constraint(model.BRANCH, rule=branchLimits_rule)

# DC line flow equation
def lineFlowEqs_rule(m, k):
    return m.pk[k] / BaseMW == (
        m.theta[branch_data[k]['from']] - m.theta[branch_data[k]['to']]
    ) / m.branch_x[k]
model.lineFlowEqs = Constraint(model.BRANCH, rule=lineFlowEqs_rule)

# Nodal power balance
def NodalPowerBalance_rule(m, n):
    gen_p    = sum(m.Pg[g]  for g in m.GEN    if m.gen_bus[g]            == n)
    flow_in  = sum(m.pk[k]  for k in m.BRANCH if branch_data[k]['to']    == n)
    flow_out = sum(m.pk[k]  for k in m.BRANCH if branch_data[k]['from']  == n)
    load = {2: total_load}.get(n, 0)
    return gen_p + flow_in - flow_out == load
model.NodalPowerBalance = Constraint(model.BUS, rule=NodalPowerBalance_rule)

model.theta[refBus].fix(0)


In [ ]:
solver = SolverFactory('gurobi')
solver.options['mipgap'] = 0.0
solver.options['timelimit'] = 90

start_time = time.time()
results = solver.solve(model, tee=True)
solve_time = time.time() - start_time

print('\n=== Generator Dispatch (Piecewise) ===')
for g in model.GEN:
    segs = [f's{s}={model.Pg_seg[g,s].value:.2f}' for s in model.SEG]
    print(f'  G[{g}] = {model.Pg[g].value:.4f} MW  ({" + ".join(segs)})')

print('\n=== Branch Flows ===')
for k in model.BRANCH:
    print(f'  pk[{k}] (Bus {branch_data[k]["from"]}→{branch_data[k]["to"]}) = {model.pk[k].value:.4f} MW')

print('\n=== Nodal LMPs ($/MWh) ===')
for n in model.BUS:
    lmp = model.dual[model.NodalPowerBalance[n]]
    print(f'  LMP[Bus {n}] = {lmp:.4f}')

total_cost = sum(gen_data[g]['seg_cost'][s-1] * model.Pg_seg[g, s].value
                 for g in model.GEN for s in model.SEG)
print(f'\n=== Total Cost = ${total_cost:.2f} ===')
print(f'Total solve elapsed time: {solve_time:.4f} seconds')
